# Sorteio de amigo secreto (versão notebook)

> Prefere não usar Python? Use a interface web: **https://serrajordia.github.io/amigo_secreto/**

Este notebook usa o pacote `amigo_secreto` deste repositório:

- sorteio com regras (ninguém se tira, ninguém tira alguém do próprio grupo familiar, corrente única);
- verificação prévia: avisa se as regras tornam o sorteio impossível, em vez de travar;
- envio pelo seu Gmail, sem que o resultado apareça na tela.

⚠️ **Nunca escreva sua senha neste notebook.** Ela vai para um arquivo `.env`, que o git ignora.

## 1. Credenciais do Gmail (uma vez só)

1. Ative a verificação em 2 etapas e crie uma **senha de app** em https://myaccount.google.com/apppasswords
2. Copie `.env.example` para `.env` e preencha `GMAIL_USER` e `GMAIL_APP_PASSWORD`.
   - Alternativa sem arquivo: `python -m amigo_secreto configurar` (guarda no cofre do Windows; requer `pip install keyring`).
3. Instale as dependências: `pip install -r requirements.txt`

In [ ]:
from amigo_secreto import Participante, Regras, sortear, verificar
from amigo_secreto.cli import ler_csv
from amigo_secreto.config import carregar_credenciais
from amigo_secreto.envio import Evento, enviar_resultados, enviar_teste

cred = carregar_credenciais()
cred  # mostra só o e-mail; a senha aparece como ***

In [ ]:
# Opcional: envia um e-mail de teste para você mesmo
enviar_teste(cred)

## 2. Participantes

Use um CSV (`nome,email,grupo` — veja `docs/modelo_participantes.csv`) ou escreva a lista aqui.
O grupo é opcional; pessoas do mesmo grupo não se tiram (se a regra estiver ligada).

In [ ]:
participantes = ler_csv("docs/modelo_participantes.csv")  # troque pelo seu arquivo

# ou manualmente:
# participantes = [
#     Participante("Ana", "ana@exemplo.com", "Silva"),
#     Participante("Bruno", "bruno@exemplo.com", "Silva"),
#     Participante("Carla", "carla@exemplo.com", "Souza"),
# ]
len(participantes)

## 3. Regras e verificação

In [ ]:
regras = Regras(
    permitir_autossorteio=False,  # ninguém tira a si mesmo
    permitir_mesmo_grupo=False,   # ninguém tira alguém do próprio grupo
    corrente_unica=False,         # True = uma roda só A→B→C→…→A
)
v = verificar(participantes, regras)
print("Viável!" if v.viavel else "Impossível:", *v.erros, *v.avisos, sep="
")

## 4. Sortear e enviar

O resultado fica na variável `destino` mas **não é exibido**. Não imprima essa variável se quiser participar sem saber! 😉

In [ ]:
evento = Evento(
    nome="Amigo Secreto da Família",
    data_festa="24/12 às 20h",
    local="Casa da vovó",
    valor_sugerido="R$ 100",
    mensagem="Levem uma sobremesa!",
)
destino = sortear(participantes, regras)
print(f"Sorteio pronto para {len(destino)} pessoas.")

In [ ]:
enviar_resultados(cred, participantes, destino, evento)